# User Question Bank Eval

Notebook นี้ไว้ดูชุดคำถาม 400 ข้อแบบรายข้อ และรันคำถามเข้า chatbot pipeline เพื่อเช็กว่าแต่ละข้อได้คำตอบจาก `fastpath/rulebase`, `rag/vector`, `rag/hybrid`, `llm` หรือ `no_answer` พร้อมดู route, source และเวลาที่ใช้

วิธีใช้เร็วๆ:
1. Run cell `Setup`
2. Run cell `Load question bank`
3. ปรับค่าใน cell `Run selected questions`
4. ดูผลรายข้อในตาราง และใช้ `show_answer("GR-001")` เพื่อเปิดคำตอบเต็ม

In [1]:
# Setup
from __future__ import annotations

import json
import sys
import time
from collections import Counter
from dataclasses import asdict, is_dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import pandas as pd
except Exception:
    pd = None

from tools.run_user_question_bank_eval import build_question_bank, _sources, _strategy
from app.runtime.pipeline_answer import answer_question_pipeline_debug

EVAL_DIR = ROOT / "data" / "eval"
BANK_PATH = EVAL_DIR / "user_question_bank_400.jsonl"
RUNS_DIR = EVAL_DIR / "question_bank_runs"

def plain(value: Any) -> Any:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(v) for v in value]
    return value

def show_table(rows: list[dict[str, Any]], columns: list[str] | None = None):
    if pd is not None:
        return pd.DataFrame(rows, columns=columns)
    if columns:
        return [{key: row.get(key, "") for key in columns} for row in rows]
    return rows

print("ROOT =", ROOT)
print("BANK_PATH =", BANK_PATH)

ROOT = c:\Users\Chokhun\Downloads\Learn-LLM\18_PSU_Esports_Update_Route_Data
BANK_PATH = c:\Users\Chokhun\Downloads\Learn-LLM\18_PSU_Esports_Update_Route_Data\data\eval\user_question_bank_400.jsonl


In [2]:
# Load question bank
bank = build_question_bank()
EVAL_DIR.mkdir(parents=True, exist_ok=True)
BANK_PATH.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in bank) + "\n", encoding="utf-8")

counts = Counter(row["category"] for row in bank)
print("total", len(bank))
print(dict(counts))

question_df = show_table(
    bank,
    ["id", "category", "question_no", "question", "expected_support", "note"],
)
question_df

total 400
{'game_rules': 100, 'play_booking_controls': 100, 'equipment_game_inside': 100, 'out_of_scope': 100}


,id,category,question_no,question,expected_support,note
0,GR-001,game_rules,1,ROV คือเกมอะไร,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
1,GR-002,game_rules,2,ROV เป็นเกมแนวไหน,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
2,GR-003,game_rules,3,ROV มีข้อมูลกติกาการแข่งขันไหม,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
3,GR-004,game_rules,4,ROV แข่งขันใช้ผู้เล่นกี่คน,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
4,GR-005,game_rules,5,ROV มีตัวสำรองได้ไหม,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
...,...,...,...,...,...,...
395,OOS-096,out_of_scope,96,ทำ infographic ด้วยเครื่องมืออะไร,general_llm_or_decline,
396,OOS-097,out_of_scope,97,เพลงฮิตตอนนี้มีอะไรบ้าง,general_llm_or_decline,
397,OOS-098,out_of_scope,98,หนังน่าดูปีนี้มีเรื่องอะไร,general_llm_or_decline,
398,OOS-099,out_of_scope,99,ช่วยคิดเมนูอาหารเย็น,general_llm_or_decline,


In [3]:
# Filter / search question bank
CATEGORY = "all"  # all, game_rules, play_booking_controls, equipment_game_inside, out_of_scope
SEARCH = ""       # เช่น ROV, Minecraft, จอง, ปุ่ม, PS5
START = 1
LIMIT = 30

filtered = bank
if CATEGORY != "all":
    filtered = [row for row in filtered if row["category"] == CATEGORY]
if SEARCH.strip():
    keyword = SEARCH.strip().lower()
    filtered = [row for row in filtered if keyword in row["question"].lower() or keyword in row["id"].lower()]

filtered = filtered[START - 1: START - 1 + LIMIT]
show_table(filtered, ["id", "category", "question_no", "question", "expected_support", "note"])

,id,category,question_no,question,expected_support,note
0,GR-001,game_rules,1,ROV คือเกมอะไร,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
1,GR-002,game_rules,2,ROV เป็นเกมแนวไหน,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
2,GR-003,game_rules,3,ROV มีข้อมูลกติกาการแข่งขันไหม,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
3,GR-004,game_rules,4,ROV แข่งขันใช้ผู้เล่นกี่คน,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
4,GR-005,game_rules,5,ROV มีตัวสำรองได้ไหม,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
5,GR-006,game_rules,6,ROV ถ้ามาสายจะเกิดอะไรขึ้น,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
6,GR-007,game_rules,7,ROV ถ้าเกมหลุดระหว่างแข่งต้องทำยังไง,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
7,GR-008,game_rules,8,ROV ขอ pause ระหว่างแข่งได้ไหม,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
8,GR-009,game_rules,9,ROV มีบทลงโทษอะไรบ้าง,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...
9,GR-010,game_rules,10,ROV ใช้โปรแกรมช่วยเล่นได้ไหม,competition_rules,มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog...


In [4]:
# Run selected questions
# ปรับตรงนี้ก่อนกด run
RUN_CATEGORY = "game_rules"  # all, game_rules, play_booking_controls, equipment_game_inside, out_of_scope
RUN_START = 1
RUN_LIMIT = 10
ALLOW_LLM = False
RAG_FALLBACK = True

def select_questions(category: str = "all", start: int = 1, limit: int | None = 10):
    rows = bank
    if category != "all":
        rows = [row for row in rows if row["category"] == category]
    rows = rows[start - 1:]
    if limit is not None:
        rows = rows[:limit]
    return rows

def run_questions(category: str = "all", start: int = 1, limit: int | None = 10, allow_llm: bool = False, rag_fallback: bool = True):
    selected = select_questions(category, start, limit)
    rows: list[dict[str, Any]] = []
    for position, item in enumerate(selected, 1):
        question = item["question"]
        started = time.perf_counter()
        result = answer_question_pipeline_debug(
            question,
            experimental_rag_fallback=rag_fallback,
            experimental_allow_llm=allow_llm,
        )
        wall_sec = round(time.perf_counter() - started, 4)
        answer = result.answer or ""
        row = {
            **item,
            "run_position": position,
            "mode": result.mode,
            "route": f"{result.route.category}/{result.route.intent}",
            "strategy": _strategy(result.mode, result.trace),
            "latency_sec": result.elapsed,
            "wall_sec": wall_sec,
            "sources": " | ".join(_sources(result.hits)),
            "answer": answer,
            "answer_preview": answer.replace("\n", " ")[:260],
            "trace": plain(result.trace),
            "validation": plain(result.validation),
        }
        rows.append(row)
        print(f"[{position}/{len(selected)}] {row['id']} | {row['strategy']} | {row['mode']} | {wall_sec}s")
    return rows

results = run_questions(RUN_CATEGORY, RUN_START, RUN_LIMIT, ALLOW_LLM, RAG_FALLBACK)
show_table(results, ["id", "category", "question", "mode", "route", "strategy", "latency_sec", "wall_sec", "sources", "answer_preview"])

[1/10] GR-001 | fastpath/rulebase | pipeline:game_detail_fast_path | 0.9147s
[2/10] GR-002 | fastpath/rulebase | pipeline:related_guidance_fast_path | 0.1589s
[3/10] GR-003 | pipeline | pipeline:competition_fact_card | 0.8315s
[4/10] GR-004 | pipeline | pipeline:competition_fact_card | 0.2376s
[5/10] GR-005 | pipeline | pipeline:competition_fact_card | 0.2027s
[6/10] GR-006 | pipeline | pipeline:competition_fact_card | 0.2921s
[7/10] GR-007 | pipeline | pipeline:competition_fact_card | 0.4542s
[8/10] GR-008 | pipeline | pipeline:competition_fact_card | 0.2859s
[9/10] GR-009 | pipeline | pipeline:rag_direct_curated | 0.4074s
[10/10] GR-010 | fastpath/rulebase | pipeline:games_known_unsupported_fast_path | 0.2583s


,id,category,question,mode,route,strategy,latency_sec,wall_sec,sources,answer_preview
0,GR-001,game_rules,ROV คือเกมอะไร,pipeline:game_detail_fast_path,games/game_availability_lookup,fastpath/rulebase,0.8684,0.9147,competition_rules: data/competition_rules,RoV / Arena of Valor: RoV หรือ Arena of Valor ...
1,GR-002,game_rules,ROV เป็นเกมแนวไหน,pipeline:related_guidance_fast_path,equipment/related_guidance,fastpath/rulebase,0.1589,0.1589,home: https://esports.phuket.psu.ac.th/home | ...,สรุปแนวเกมที่มีข้อมูลยืนยันได้: - FPS/Tactical...
2,GR-003,game_rules,ROV มีข้อมูลกติกาการแข่งขันไหม,pipeline:competition_fact_card,competition_rules/competition_rules_lookup,pipeline,0.8315,0.8315,rov_team_size_active_players: local://competit...,คำตอบ: ไฟล์กติกา RoV ระบุว่าแข่งขันในโหมด 5v5 ...
3,GR-004,game_rules,ROV แข่งขันใช้ผู้เล่นกี่คน,pipeline:competition_fact_card,competition_rules/competition_rules_lookup,pipeline,0.2376,0.2376,rov_team_size_active_players: local://competit...,คำตอบ: ไฟล์กติกา RoV ระบุว่าแข่งขันในโหมด 5v5 ...
4,GR-005,game_rules,ROV มีตัวสำรองได้ไหม,pipeline:competition_fact_card,competition_rules/competition_rules_lookup,pipeline,0.2026,0.2027,rov_team_size_active_players: local://competit...,คำตอบ: ไฟล์กติกา RoV ระบุว่าแข่งขันในโหมด 5v5 ...
5,GR-006,game_rules,ROV ถ้ามาสายจะเกิดอะไรขึ้น,pipeline:competition_fact_card,competition_rules/competition_rules_lookup,pipeline,0.2920,0.2921,rov_late_start_forfeit: local://competition_ru...,คำตอบ: RoV ถ้าเริ่มแข่งล่าช้าเกิน 15 นาที จะถู...
6,GR-007,game_rules,ROV ถ้าเกมหลุดระหว่างแข่งต้องทำยังไง,pipeline:competition_fact_card,competition_rules/competition_rules_lookup,pipeline,0.4541,0.4542,rov_pause_disconnect: local://competition_rule...,คำตอบ: RoV แต่ละทีมสามารถหยุดเกมได้สูงสุด 5 คร...
7,GR-008,game_rules,ROV ขอ pause ระหว่างแข่งได้ไหม,pipeline:competition_fact_card,competition_rules/competition_rules_lookup,pipeline,0.2858,0.2859,rov_pause_disconnect: local://competition_rule...,คำตอบ: RoV แต่ละทีมสามารถหยุดเกมได้สูงสุด 5 คร...
8,GR-009,game_rules,ROV มีบทลงโทษอะไรบ้าง,pipeline:rag_direct_curated,competition_rules/competition_rules_lookup,pipeline,0.4073,0.4074,competition_rules_rov_blueket_2025_men_s06_c04...,คำตอบ: 6.1.2.1. บทลงโทษ: ปรับแพ้ในเกมที่พบการก...
9,GR-010,game_rules,ROV ใช้โปรแกรมช่วยเล่นได้ไหม,pipeline:games_known_unsupported_fast_path,games/game_availability_lookup,fastpath/rulebase,0.2583,0.2583,our_games: https://esports.phuket.psu.ac.th/Se...,ยังไม่พบ RoV / Arena of Valor ในรายการเกมที่ยื...


In [5]:
# ดูคำตอบเต็มรายข้อ
def show_answer(question_id: str):
    matches = [row for row in results if row["id"] == question_id]
    if not matches:
        print("ไม่พบ id นี้ใน results ที่รันล่าสุด:", question_id)
        return None
    row = matches[0]
    print("ID:", row["id"])
    print("Category:", row["category"])
    print("Question:", row["question"])
    print("Mode:", row["mode"])
    print("Route:", row["route"])
    print("Strategy:", row["strategy"])
    print("Latency:", row["latency_sec"], "| Wall:", row["wall_sec"])
    print("Sources:", row["sources"] or "-")
    print("\nAnswer:\n", row["answer"])
    return row

show_answer(results[0]["id"]) if results else None

ID: GR-001
Category: game_rules
Question: ROV คือเกมอะไร
Mode: pipeline:game_detail_fast_path
Route: games/game_availability_lookup
Strategy: fastpath/rulebase
Latency: 0.8684 | Wall: 0.9147
Sources: competition_rules: data/competition_rules

Answer:
 RoV / Arena of Valor: RoV หรือ Arena of Valor คือเกม MOBA บนมือถือที่ผู้เล่นแบ่งเป็นทีม เลือกฮีโร่ และร่วมกันทำลายป้อม/ฐานของฝ่ายตรงข้าม
แนวเกม: เกม MOBA แบบทีม
วิธีเล่นโดยสรุป: โดยทั่วไปผู้เล่นต้องเลือกตำแหน่งและฮีโร่ให้เหมาะกับทีม เก็บเลเวล คุมแผนที่ ช่วยทีมไฟต์ และดันเลนเพื่อทำลายฐานคู่แข่ง ในฐานข้อมูลของศูนย์มีข้อมูลฝั่งกติกาการแข่งขัน แต่ยังไม่พบว่าอยู่ในรายการเกมให้เล่นของศูนย์
เล่นได้ที่: มีข้อมูลกติกาการแข่งขัน แต่ยังไม่พบในรายการเกมให้เล่นของศูนย์
แหล่งข้อมูล: data/competition_rules


{'id': 'GR-001',
 'category': 'game_rules',
 'question_no': 1,
 'question': 'ROV คือเกมอะไร',
 'expected_support': 'competition_rules',
 'note': 'มีข้อมูลกติกาการแข่งขัน แต่ไม่ใช่เกมใน catalog เล่นของศูนย์',
 'run_position': 1,
 'mode': 'pipeline:game_detail_fast_path',
 'route': 'games/game_availability_lookup',
 'strategy': 'fastpath/rulebase',
 'latency_sec': 0.8684,
 'wall_sec': 0.9147,
 'sources': 'competition_rules: data/competition_rules',
 'answer': 'RoV / Arena of Valor: RoV หรือ Arena of Valor คือเกม MOBA บนมือถือที่ผู้เล่นแบ่งเป็นทีม เลือกฮีโร่ และร่วมกันทำลายป้อม/ฐานของฝ่ายตรงข้าม\nแนวเกม: เกม MOBA แบบทีม\nวิธีเล่นโดยสรุป: โดยทั่วไปผู้เล่นต้องเลือกตำแหน่งและฮีโร่ให้เหมาะกับทีม เก็บเลเวล คุมแผนที่ ช่วยทีมไฟต์ และดันเลนเพื่อทำลายฐานคู่แข่ง ในฐานข้อมูลของศูนย์มีข้อมูลฝั่งกติกาการแข่งขัน แต่ยังไม่พบว่าอยู่ในรายการเกมให้เล่นของศูนย์\nเล่นได้ที่: มีข้อมูลกติกาการแข่งขัน แต่ยังไม่พบในรายการเกมให้เล่นของศูนย์\nแหล่งข้อมูล: data/competition_rules',
 'answer_preview': 'RoV / Arena of

In [6]:
# Save notebook run results
def save_results(rows: list[dict[str, Any]], label: str = "notebook"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = RUNS_DIR / f"{timestamp}_{label}"
    out_dir.mkdir(parents=True, exist_ok=True)
    jsonl_path = out_dir / "results.jsonl"
    jsonl_path.write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in rows) + "\n", encoding="utf-8")
    (out_dir / "results.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    by_category = {}
    for row in rows:
        by_category.setdefault(row.get("category", "unknown"), []).append(row)
    (out_dir / "results_by_category.json").write_text(json.dumps(by_category, ensure_ascii=False, indent=2), encoding="utf-8")
    summary = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "total": len(rows),
        "strategy_counts": dict(Counter(row["strategy"] for row in rows)),
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved to", out_dir)
    return out_dir

# save_results(results)

In [7]:
# Load latest saved run from CLI/notebook
def load_latest_run():
    if not RUNS_DIR.exists():
        print("ยังไม่มี question_bank_runs")
        return []
    run_dirs = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()], key=lambda p: p.name, reverse=True)
    if not run_dirs:
        print("ยังไม่มี run directory")
        return []
    latest = run_dirs[0]
    json_path = latest / "results.json"
    jsonl_path = latest / "results.jsonl"
    if json_path.exists():
        rows = json.loads(json_path.read_text(encoding="utf-8"))
    elif jsonl_path.exists():
        rows = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    else:
        print("run ล่าสุดไม่มี results.json หรือ results.jsonl:", latest)
        return []
    print("loaded", latest, "rows", len(rows))
    return rows

latest_results = load_latest_run()
show_table(latest_results, ["id", "category", "question", "mode", "route", "strategy", "wall_sec", "sources", "answer"] if latest_results else None)

loaded c:\Users\Chokhun\Downloads\Learn-LLM\18_PSU_Esports_Update_Route_Data\data\eval\question_bank_runs\20260715_155252 rows 6


,id,category,question,mode,route,strategy,wall_sec,sources,answer
0,OOS-001,out_of_scope,สอนทำข้าวผัดแบบง่ายๆ,pipeline:general_llm_disabled,general/general_knowledge_query,llm,0.5942,,คำถามนี้ถูกจัดเป็นคำถามทั่วไปนอกฐานข้อมูล PSU ...
1,OOS-002,out_of_scope,วันนี้ฝนจะตกไหม,pipeline:calendar_schedule_fast_path,schedule/schedule_query,fastpath/rulebase,1.2214,Reservation: https://esports.computing.psu.ac....,วันนี้ 15/07/2026 (วันพุธ): วันพุธเปิดให้เล่น ...
2,OOS-003,out_of_scope,ช่วยแปลประโยคนี้เป็นอังกฤษได้ไหม,pipeline:general_llm_disabled,general/unknown_domain_query,llm,0.3711,,คำถามนี้ถูกจัดเป็นคำถามทั่วไปนอกฐานข้อมูล PSU ...
3,OOS-004,out_of_scope,สูตรคำนวณพื้นที่วงกลมคืออะไร,pipeline:general_llm_disabled,general/general_knowledge_query,llm,0.2659,,คำถามนี้ถูกจัดเป็นคำถามทั่วไปนอกฐานข้อมูล PSU ...
4,OOS-005,out_of_scope,ขอไอเดียตั้งชื่อร้านกาแฟ,pipeline:general_llm_disabled,general/unknown_domain_query,llm,0.1991,,คำถามนี้ถูกจัดเป็นคำถามทั่วไปนอกฐานข้อมูล PSU ...
5,OOS-006,out_of_scope,ช่วยเขียนคำอวยพรวันเกิดให้เพื่อน,pipeline:experimental_soft_related_fallback,general/unknown_domain_query,fastpath/rulebase,0.3877,reservation: https://esports.computing.psu.ac....,โหมดทดลอง RAG: ข้อมูลที่ยืนยันได้คือศูนย์มีระบ...
